# 01 Data Check

Use this notebook to verify that every supported dataset loads correctly, has the expected shape, and can be split into train/validation/test sets.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
from IPython.display import display

from src.data.load_datasets import load_dataset
from src.data.preprocessing import make_train_val_test_split

DATASETS = ['breast_cancer', 'wine', 'digits', 'dry_bean']
RANDOM_STATE = 42

## Dataset Overview

In [ ]:
overview_rows = []

for dataset_name in DATASETS:
    dataset = load_dataset(dataset_name)
    overview_rows.append(
        {
            'dataset': dataset.name,
            'samples': len(dataset.X),
            'features': dataset.X.shape[1],
            'classes': dataset.n_classes,
            'task_type': dataset.task_type,
        }
    )

dataset_overview = pd.DataFrame(overview_rows)
display(dataset_overview)

## Class Balance

In [ ]:
for dataset_name in DATASETS:
    dataset = load_dataset(dataset_name)
    counts = (
        dataset.y.value_counts(normalize=False)
        .rename_axis('class')
        .reset_index(name='count')
        .sort_values('class')
    )
    counts['fraction'] = counts['count'] / counts['count'].sum()
    print(dataset_name)
    display(counts)

## Split Check

In [ ]:
split_rows = []

for dataset_name in DATASETS:
    dataset = load_dataset(dataset_name)
    X_train, X_val, X_test, y_train, y_val, y_test = make_train_val_test_split(
        dataset.X,
        dataset.y,
        random_state=RANDOM_STATE,
    )
    split_rows.append(
        {
            'dataset': dataset_name,
            'train_samples': len(X_train),
            'val_samples': len(X_val),
            'test_samples': len(X_test),
            'train_classes': y_train.nunique(),
            'val_classes': y_val.nunique(),
            'test_classes': y_test.nunique(),
        }
    )

split_check = pd.DataFrame(split_rows)
display(split_check)

## Optional Save

In [ ]:
output_path = Path('results/tables/notebook_dataset_check.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
dataset_overview.to_csv(output_path, index=False)
print(f'Saved {output_path}')